# Conjugate gradient: Gram-Schmidt orthogonalisation and the classical algorithm

In [ ]:
#    APM41012EP course notebook - Chapter 5 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    CG: Gram-Schmidt orthogonalisation and the classical algorithm
#    Test on the Hilbert matrix with an exact solve using SymPy
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import sympy as sp

We want to solve the linear system $A\,V = b$ by an iterative method of conjugate gradient type.


Initialisation: 
- $ V^0 = 0 $
- $ r^0 = b $
- $ p^0 = r^0 $ the usual gradient descent direction for the first step.

We want to show that iteration $k$ of the conjugate gradient algorithm with the Gram-Schmidt orthogonalisation introduced in the lecture:

Iteration $k$: 
- $ V^{k+1} = V^k + \alpha^k p^k$ with $\displaystyle \alpha^k = \frac{p^k r^k}{p^k Ap^k}$
- $ r^{k+1} = b - A\,V^{k+1}$
- $ \displaystyle p^{k+1} = r^{k+1} - \sum_{i=0}^{k} \frac{p^i Ar^k}{p^i Ap^i} p^i$ 

is mathematically equivalent to the formulation of the algorithm that is generally found:

- $ V^{k+1} = V^k + \alpha^k p^k$ with $\displaystyle \alpha^k = \frac{r^k r^k}{p^k Ap^k}$
- $ r^{k+1} = r^k - \alpha^k A p^k$
- $ p^{k+1} = r^{k+1} + \beta^k p^k$ with $\displaystyle \beta^k = \frac{r^{k+1} r^{k+1}}{r^{k} r^{k}}$


**Remark:** The directions $p^k$ are $A$-conjugate (Exercise: prove it by induction).

We define the Krylov space $\mathcal{K}_k(A, r^0) = \textrm{Vect}\{r^0, Ar^0, \dots, A^{k-1} r^0\}$,
for which it is easy to see that $\mathcal{K}_k(A, r^0) = \textrm{Vect}\{r^0, r^1, \dots, r^{k-1}\}$.

Because of the construction of the iterate $V^k$, one can show that $V^k$ equivalently solves the two minimisation problems:

$$
\min_{V \in V_0 + \mathcal{K}_k(A, r^0)} \| V - V^* \|_A, \quad \min_{V \in V_0 + \mathcal{K}_k(A, r^0)} \phi(V),
$$
where $\phi$ is the quadratic form introduced in the lecture, and 
thus obtain a certain number of properties: 
$$
V^k\in V^0+\mathcal{K}_k(A, r^0), \quad r^k\perp \mathcal{K}_k(A, r^0)
$$
$$
p^k\in\mathcal{K}_{k+1}(A, r^0), \quad p^k\perp_A \mathcal{K}_k(A, r^0)
$$

The interested reader may consult C.T. Kelley *Iterative Methods for Linear and Nonlinear Equations* (1995)
SIAM 

**Lemma**

$$
\left\{
\begin{aligned}
r^{k+1} & = r^k - \alpha^k A p^k\\
r^{k} A p^k & = p^{k} A p^k\\
p^{k}r^k & = r^k r^k
\end{aligned}
\right.
$$

Proof:

$r^{k+1} = b - A V^{k+1} = b - A (V^k + \alpha^k p^k) = r^k - \alpha^k A p^k$

moreover $ \displaystyle p^{k} = r^{k} - \sum_{i=0}^{k-1} \frac{p^i Ar^k}{p^i Ap^i} p^i$ and $p^kAp^k = r^kAp^k$ since all the other terms vanish by $A$-conjugation.

Finally, we have ${r^{k+1}}^t r^k=0$ and therefore

$${r^k}^t r^k= \frac{p^k r^k}{p^k Ap^k} r^k A p^k$$

and we conclude with the equality above $r^{k} A p^k  = p^{k} A p^k$.


**Lemma**

$$
\left\{
\begin{aligned}
\alpha^k & = \frac{r^k r^k}{p^k Ap^k}\\
p^{k+1} & = r^{k+1} + \beta^k p^k,\quad \displaystyle \beta^k = \frac{r^{k+1} r^{k+1}}{r^{k} r^{k}}
\end{aligned}
\right.
$$

Proof:

The first equality comes from the previous lemma and, concerning the second one, we first have: 

$$
{r^{k+1}}^t r^{k+1}= - {r^k}^t r^k \frac{p^k A r^{k+1}}{p^k Ap^k}
$$

by taking the scalar product of the equality obtained above with $r^{k+1}$.

Moreover, by taking the scalar product of $r^{i+1} = r^i - \alpha^i A p^i$ with $r^{k+1}$ for $i\le k-1$, we see that 

$$
p^i A r^{k+1} = 0, \quad i\le k-1,
$$

and therefore

$$
p^{k+1}  = r^{k+1} - \frac{p^k A r^{k+1}}{p^k Ap^k} p^k, 
$$

and by replacing the factor of $p^k$ by its expression, we indeed obtain $p^{k+1}  = r^{k+1} + \beta^k p^k$.


In [ ]:
def conjugate_gradient(a, b, eps=1.e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        ####print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")
        if norm_rk/norm_b < eps: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    print(f"  Number of iterations = {k+1}")
    print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")

    return xk

def conjugate_gradient_gram_schmidt(a, b, eps=1.e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    hist_pk = []
    hist_apk = []

    hist_pk.append(pk)

    for k in range(b.size):
        apk = a.dot(pk)
        hist_apk.append(apk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        ####print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")
        if norm_rk/norm_b < eps: break
        betap = np.zeros(b.size)
        for m in range(k+1):
            beta = - np.dot(rk, hist_apk[m]) / np.dot(hist_pk[m], hist_apk[m])
            betap = betap + beta * hist_pk[m]
        pk = rk + betap
        hist_pk.append(pk)

    print(f"  Number of iterations = {k+1}")
    print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")

    return xk

## Test with the Hilbert matrix

In [ ]:
n = 7

print("Size of the Hilbert matrix: ", n)
print()

# Hilbert matrix with EXACT rational entries
A = sp.Matrix(n, n, lambda i, j: sp.Rational(1, i + j + 1))
cond_2 = np.linalg.norm(np.array(A.tolist(), dtype=np.float64), 2) * np.linalg.norm(np.array(A.inv().tolist(), dtype=np.float64), 2)
print("Condition number associated with the 2-norm:", cond_2)
print()

b = sp.Matrix([1 for i in range(n)])

x = A.solve(b)          # EXACT (rational) solve
print("Exact solution obtained with SymPy")
sp.pprint(x.T, wrap_line=False)

A1 = np.array(A.tolist(), dtype=np.float64)
b1 = np.ones(n)
xe = np.array(x.tolist(), dtype=np.float64).ravel()

print("\nSolution with the classical conjugate gradient")
x1 = conjugate_gradient(A1, b1, eps=1.e-6)
np.set_printoptions(precision=12, suppress=True, linewidth=150)
if (n<11):
    print("  Solution:")
    print(x1)
print(f"  Norm of the error: {np.linalg.norm(xe-x1):.5e}")


print("\nSolution with the conjugate gradient in its Gram-Schmidt form")
x1 = conjugate_gradient_gram_schmidt(A1, b1, eps=1.e-6)
if (n<11):
    print("  Solution:")
    print(x1)
print(f"  Norm of the error: {np.linalg.norm(xe-x1):.5e}")